# Reporte de evaluación — Banco Copilot

Lee los resultados de la última corrida (`evals/results.jsonl`) y el conjunto de referencia (`evals/dataset.jsonl`). **No ejecuta la evaluación**; la corrida se hace con `uv run --all-packages python -m evals.run --target <url>`, que además la sube a Foundry → Evaluations.

Se abre desde `evals/notebooks/`: las rutas son relativas a esa carpeta.

In [ ]:
import json
from pathlib import Path

import pandas as pd

EVALS_DIR = Path.cwd().parent


def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]


dataset = {row["id"]: row for row in read_jsonl(EVALS_DIR / "dataset.jsonl")}
results = read_jsonl(EVALS_DIR / "results.jsonl")
f"{len(results)} filas evaluadas de {len(dataset)} en el dataset"

## Filas

Una fila por pregunta: estado esperado y obtenido, latencia, tokens, costo, puntajes de los jueces de Foundry (1 a 5) y la traza resumida: ruta del router, tools llamadas y lo que midió el verificador.

In [ ]:
def status(result):
    return result["error"]["code"] if result["error"] else result["answer"]["status"]


def trace_summary(traces):
    parts = []
    for event in traces:
        data = event["data"]
        if event["node"] == "router":
            parts.append(f"router: {data['route']}")
        elif event["node"] == "verifier":
            parts.append(f"verificador: sin sustento {data['ungrounded_percentage']}, "
                         f"sin cita {data['uncited_claims']}, contradicción {data['contradiction']}")
        elif data.get("tool_calls"):
            parts.append(f"{event['node']}: " + ", ".join(call["tool"] for call in data["tool_calls"]))
        if "guardrail" in data:
            parts.append(f"guardrail: {data['guardrail']}")
    return " | ".join(parts)


def verifier_ungrounded(traces):
    shares = [t["data"]["ungrounded_percentage"] for t in traces if t["node"] == "verifier"]
    return shares[-1] if shares else None


rows = pd.DataFrame([
    {
        "id": r["id"],
        "tipo": dataset[r["id"]]["type"],
        "esperado": dataset[r["id"]]["expected_status"],
        "obtenido": status(r),
        "latencia_ms": r["latency_ms"],
        "tokens": sum(t["tokens_in"] + t["tokens_out"] for t in r["traces"]),
        "costo_usd": sum(t["cost_usd"] for t in r["traces"]),
        **r["scores"],
        "verificador_sin_sustento": verifier_ungrounded(r["traces"]),
        "traza": trace_summary(r["traces"]),
    }
    for r in results
]).set_index("id")
rows["acierto"] = rows["esperado"] == rows["obtenido"]
rows.drop(columns="traza")

## Resultados por tipo de pregunta

In [ ]:
rows.groupby("tipo").agg(
    filas=("acierto", "size"),
    aciertos=("acierto", "sum"),
    exactitud=("acierto", "mean"),
    latencia_p50_ms=("latencia_ms", "median"),
    costo_usd=("costo_usd", "sum"),
    groundedness=("groundedness", "mean"),
).round(3)

## Matriz esperado / obtenido

Filas: estado esperado. Columnas: estado obtenido (o código de error). La diagonal son los aciertos.

In [ ]:
pd.crosstab(rows["esperado"], rows["obtenido"], margins=True, margins_name="total")

## Latencia y costo

Las filas con error de transporte tienen latencia 0 y se excluyen de los percentiles.

In [ ]:
answered = rows[rows["latencia_ms"] > 0]
pd.Series({
    "latencia p50 (ms)": answered["latencia_ms"].quantile(0.5),
    "latencia p95 (ms)": answered["latencia_ms"].quantile(0.95),
    "tokens promedio": rows["tokens"].mean(),
    "costo total (USD)": rows["costo_usd"].sum(),
    "costo por pregunta (USD)": rows["costo_usd"].mean(),
}).round(4)

## Groundedness por fila

`groundedness` es el juez de Foundry (1 a 5), y solo puntúa respuestas `ok` con contexto recuperado. `verificador_sin_sustento` es la fracción que el verificador del copiloto (Content Safety) marcó sin sustento en vivo.

In [ ]:
rows[["tipo", "obtenido", "groundedness", "verificador_sin_sustento"]]

## Fallos

Filas donde el estado obtenido no es el esperado, con su traza resumida.

In [ ]:
with pd.option_context("display.max_colwidth", None, "display.width", 250):
    print(rows.loc[~rows["acierto"], ["tipo", "esperado", "obtenido", "traza"]].to_string())